# Accessing Resources

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_client.py`; the finished version is in `cli-project-complete/mcp_client.py`.

The server exposes resources (last lesson); now the **client** reads them. Resources let you **inject data straight into the prompt** rather than making Claude call a tool for it — faster and cheaper for context you already know you want. This powers the `@document_name` **mention** feature: type `@`, pick a doc (from the `docs://documents` list), and its contents (from `docs://documents/{doc_id}`) get pasted into the message to Claude.

## What to modify

**File:** `cli-project/mcp_client.py`

**1. Add imports** (top of file):
```python
import json
from pydantic import AnyUrl
```

**2. Implement `read_resource`** (replace the `TODO` stub):
```python
async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)

        return resource.text
```

## How it works

- **`session().read_resource(AnyUrl(uri))`** sends a `ReadResourceRequest` for that URI; the server responds with a **`contents`** list. You usually want `contents[0]`.
- **MIME-type-driven parsing:** the resource carries its `mimeType` (the hint the server set). For `application/json` (our `docs://documents` list) → `json.loads` into a Python object; for `text/plain` (our `docs://documents/{doc_id}` content) → return the string as-is.
- **`AnyUrl`** (Pydantic) gives the URI proper typing for the SDK call.

So the same method serves both resources: reading `docs://documents` yields the id list (for autocomplete), and `docs://documents/report.pdf` yields that doc's text (to inject).

## Resources vs tools (why this matters)

- **Tool call:** Claude must *decide* to call `read_doc_contents`, you execute it, send the result back — a full round-trip.
- **Resource injection:** *your app* fetches the content up front (because the user explicitly `@`-mentioned it) and puts it directly in the prompt — no tool round-trip, lower latency.

Use resources when the app already knows what data to include; use tools when Claude should decide.

## How it plugs into the CLI

`core/cli_chat.py` uses the client to power mentions: it calls `read_resource("docs://documents")` to build the autocomplete list when you type `@`, and `read_resource("docs://documents/<id>")` to fetch content for any `@mention` in your message, splicing it into the prompt before sending to Claude. You only had to implement `read_resource`; the app wiring was already there.

## Testing — run the app and mention a doc

Resource access is end-to-end (client spawns the server), so test in a **terminal**:

```powershell
.\.venv\Scripts\Activate.ps1
cd building-with-the-claude-api\07-mcp\cli-project-complete
python main.py
```

- Type **`@`** → an autocomplete list of documents appears (backed by `docs://documents`).
- Ask: **"What's in the @report.pdf document?"** → the app injects `report.pdf`'s contents into the prompt, and Claude answers from it **without** a tool call.

Run in `cli-project/` instead once you've added `read_resource` there.

## Where this leaves us

Resources are now fully wired — server exposes them, client reads them, the CLI turns them into `@`-mentions. Tools (actions) + resources (data) are both covered. The last MCP primitive is **prompts** (server-provided prompt templates):

Next: **Defining prompts** — a server-side `@mcp.prompt` that rewrites/summarizes a document.